# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SupreetOP/Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use a Random Forest model for the content opportunity scoring lane.

The Week-4 baseline is a hand-written rule that ranks content using observed Search Console signals, especially impressions and CTR. Random Forest is a suitable next step because it can learn non-linear relationships and interactions between the available content-performance features without assuming that their effects are linear.

The model will use only information available at the decision moment and will be evaluated against the Week-4 rule-based baseline on the same data and validation design. The goal is not to add complexity for its own sake, but to test whether a learned model provides better ranking or prediction performance than the simple baseline.

## 2. Split design

I use a time-aware split because the goal is to identify content opportunities using information available at the decision moment and evaluate whether those signals relate to a subsequent outcome.

March 2026 is used as the decision/feature period, and April 2026 is used as the future validation outcome period. The model therefore learns from information available in March and is evaluated against what happens in April.

I do not randomly mix March and April observations because that could allow future information to influence the training process. The Week-4 baseline will be evaluated on the same March-to-April setup so the model and baseline receive the same information and are compared fairly.

June 2026 is treated as a sealed final month and is not used for developing the model or its rules.

In [6]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET hf_secret ("
    f"TYPE HUGGINGFACE, "
    f"TOKEN '{HF_TOKEN}'"
    f")"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [7]:
months = con.sql(f"""
    SELECT
        month,
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY month
    ORDER BY month
""").df()

print("=" * 70)
print("AVAILABLE TIME WINDOWS")
print("=" * 70)

display(months)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

AVAILABLE TIME WINDOWS


,month,row_count,min_date,max_date
0,2025-01,1297,2025-01-27,2025-01-31
1,2025-02,75985,2025-02-01,2025-02-28
2,2025-03,167859,2025-03-01,2025-03-31
3,2025-04,285114,2025-04-01,2025-04-30
4,2025-05,349923,2025-05-01,2025-05-31
5,2025-06,329201,2025-06-01,2025-06-30
6,2025-07,469794,2025-07-01,2025-07-31
7,2025-08,704962,2025-08-01,2025-08-31
8,2025-09,845813,2025-09-01,2025-09-30
9,2025-10,2165471,2025-10-01,2025-10-31


In [9]:


FEATURE_MONTH = "2026-03"
OUTCOME_MONTH = "2026-04"

# March = information available at decision time
march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS gsc_avg_position,
        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# April = future outcome period
april_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_gsc_impressions,
        SUM(gsc_clicks) AS future_gsc_clicks,
        SUM(ga4_pageviews) AS future_ga4_pageviews,
        SUM(ga4_engaged_sessions) AS future_ga4_engaged_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '{OUTCOME_MONTH}'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("=" * 70)
print("TIME-AWARE SPLIT")
print("=" * 70)

print("Feature month:", FEATURE_MONTH)
print("Outcome month:", OUTCOME_MONTH)
print("March feature rows:", len(march_features))
print("April outcome rows:", len(april_outcomes))

display(march_features.head())
display(april_outcomes.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TIME-AWARE SPLIT
Feature month: 2026-03
Outcome month: 2026-04
March feature rows: 331437
April outcome rows: 362172


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,7.842593,0.0,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,8.454069,4.0,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,12.0,0.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0


,client_hash_id,content_hash_id,future_gsc_impressions,future_gsc_clicks,future_ga4_pageviews,future_ga4_engaged_sessions
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0,0.0,NaN,NaN
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0,0.0,NaN,NaN
2,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,8.0,0.0,NaN,NaN
3,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0,0.0,NaN,NaN
4,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0,1.0,NaN,NaN


## 3. Train + compare vs my baseline

The Random Forest model is trained using only features available at the decision moment. The model and the Week-4 rule-based baseline are evaluated on the same held-out validation data and using the same metric.

The comparison is intended to answer one question: does the learned model provide better content-opportunity predictions than the simple rule-based baseline?

In [11]:


from sklearn.model_selection import train_test_split

# Binary future-opportunity target:
# 1 = received at least one GSC click in April
# 0 = received zero GSC clicks in April
model_data["target"] = (
    model_data["future_gsc_clicks"] > 0
).astype(int)

# Features available at the March decision moment ONLY
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

X = model_data[feature_columns].copy()
y = model_data["target"].copy()

# Missing feature values are filled using training-safe median values.
# No April information is used here.
for col in feature_columns:
    X[col] = X[col].fillna(X[col].median())

# Keep the split reproducible.
# We stratify the binary target so both sets contain both classes.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TARGET + VALIDATION SPLIT")
print("=" * 70)

print("Target definition: April GSC clicks > 0")
print("Total rows:", len(model_data))
print("Positive targets:", y.sum())
print("Negative targets:", (y == 0).sum())

print("\nTraining rows:", len(X_train))
print("Validation rows:", len(X_valid))

print("\nTraining positive rate:", round(y_train.mean(), 4))
print("Validation positive rate:", round(y_valid.mean(), 4))

print("\nFeatures used:")
print(feature_columns)

TARGET + VALIDATION SPLIT
Target definition: April GSC clicks > 0
Total rows: 331436
Positive targets: 62098
Negative targets: 269338

Training rows: 265148
Validation rows: 66288

Training positive rate: 0.1874
Validation positive rate: 0.1874

Features used:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


In [12]:


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score
)

# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train only on March features
rf_model.fit(X_train, y_train)

# Predict probability of April clicks > 0
valid_probability = rf_model.predict_proba(X_valid)[:, 1]

# Convert probability to class prediction
valid_prediction = (valid_probability >= 0.5).astype(int)

# Metrics
roc_auc = roc_auc_score(y_valid, valid_probability)
pr_auc = average_precision_score(y_valid, valid_probability)
accuracy = accuracy_score(y_valid, valid_prediction)

print("=" * 70)
print("RANDOM FOREST VALIDATION RESULTS")
print("=" * 70)

print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")

print("\nModel trained using:")
print(feature_columns)

RANDOM FOREST VALIDATION RESULTS
ROC-AUC : 0.9312
PR-AUC  : 0.8397
Accuracy: 0.8762

Model trained using:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


In [13]:


# Recover the original validation indices
valid_indices = X_valid.index

baseline_valid = model_data.loc[
    valid_indices,
    [
        "gsc_impressions",
        "gsc_clicks",
        "target"
    ]
].copy()

# Calculate March CTR safely
baseline_valid["ctr"] = np.where(
    baseline_valid["gsc_impressions"] > 0,
    baseline_valid["gsc_clicks"] /
    baseline_valid["gsc_impressions"],
    0
)

# Week-4 rule:
# REFRESH when impressions >= 100 AND CTR <= 2%
baseline_valid["baseline_prediction"] = (
    (baseline_valid["gsc_impressions"] >= 100) &
    (baseline_valid["ctr"] <= 0.02)
).astype(int)

# Baseline probability-like score for ranking comparison
# 1 = baseline predicts future opportunity
# 0 = baseline does not
baseline_probability = baseline_valid["baseline_prediction"]

# Metrics
baseline_roc_auc = roc_auc_score(
    baseline_valid["target"],
    baseline_probability
)

baseline_pr_auc = average_precision_score(
    baseline_valid["target"],
    baseline_probability
)

baseline_accuracy = accuracy_score(
    baseline_valid["target"],
    baseline_valid["baseline_prediction"]
)

print("=" * 70)
print("WEEK-4 BASELINE — SAME VALIDATION SET")
print("=" * 70)

print(f"ROC-AUC : {baseline_roc_auc:.4f}")
print(f"PR-AUC  : {baseline_pr_auc:.4f}")
print(f"Accuracy: {baseline_accuracy:.4f}")

print("\nBaseline action counts:")
print(
    baseline_valid["baseline_prediction"]
    .map({
        0: "MONITOR",
        1: "REFRESH"
    })
    .value_counts()
)

WEEK-4 BASELINE — SAME VALIDATION SET
ROC-AUC : 0.8447
PR-AUC  : 0.4858
Accuracy: 0.8325

Baseline action counts:
baseline_prediction
MONITOR    46146
REFRESH    20142
Name: count, dtype: int64


In [14]:

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Rule Baseline",
        "Random Forest"
    ],
    "ROC-AUC": [
        baseline_roc_auc,
        roc_auc
    ],
    "PR-AUC": [
        baseline_pr_auc,
        pr_auc
    ],
    "Accuracy": [
        baseline_accuracy,
        accuracy
    ]
})

print("=" * 70)
print("MODEL VS WEEK-4 BASELINE")
print("=" * 70)

display(
    comparison.style.format({
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
        "Accuracy": "{:.4f}"
    })
)

# Absolute improvement
print("\nIMPROVEMENT OF RANDOM FOREST OVER BASELINE")
print("-" * 70)

print(
    f"ROC-AUC improvement: "
    f"{roc_auc - baseline_roc_auc:+.4f}"
)

print(
    f"PR-AUC improvement: "
    f"{pr_auc - baseline_pr_auc:+.4f}"
)

print(
    f"Accuracy improvement: "
    f"{accuracy - baseline_accuracy:+.4f}"
)

MODEL VS WEEK-4 BASELINE


,Method,ROC-AUC,PR-AUC,Accuracy
0,Week-4 Rule Baseline,0.8447,0.4858,0.8325
1,Random Forest,0.9312,0.8397,0.8762



IMPROVEMENT OF RANDOM FOREST OVER BASELINE
----------------------------------------------------------------------
ROC-AUC improvement: +0.0865
PR-AUC improvement: +0.3539
Accuracy improvement: +0.0436


### Section 3 conclusion

On the same validation data, the Random Forest performed better than the Week-4 rule-based baseline across all three reported metrics. The observed ROC-AUC increased from 0.8447 to 0.9312, PR-AUC increased from 0.4858 to 0.8397, and accuracy increased from 0.8325 to 0.8762.

The largest improvement was in PR-AUC, suggesting that the learned model separates future positive outcomes better than the simple Week-4 rule. These results are directional validation evidence rather than proof of production performance, because the evaluation uses one March-to-April period.

## 4. Errors and interpretation

The model's errors should be reviewed alongside the baseline rather than judged only by the overall metric. Particular attention should be given to cases where the model and the Week-4 baseline disagree, because these cases show where the learned model is making different decisions from the rule-based approach.

The model is expected to rely most strongly on the observed content-performance signals available at the decision moment, such as Search Console impressions, clicks, average position, and available Analytics engagement measures. Any apparent errors may occur when these signals do not fully capture the actual reason a page represents a refresh opportunity, such as search intent, content quality, seasonality, or changes in demand that are not represented in the available features.

The model should therefore be treated as decision-support rather than as proof that a page should be refreshed.

In [15]:
# ============================================================
# SECTION 4 — ERRORS AND INTERPRETATION
# ============================================================

from sklearn.metrics import confusion_matrix

print("=" * 70)
print("SECTION 4 — ERROR ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Feature importance
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("\nFEATURE IMPORTANCE")
print("-" * 70)

display(
    feature_importance.style.format({
        "importance": "{:.4f}"
    })
)

# ------------------------------------------------------------
# 2. Validation predictions
# ------------------------------------------------------------

error_analysis = model_data.loc[
    X_valid.index,
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_engaged_sessions",
        "future_gsc_clicks",
        "target"
    ]
].copy()

error_analysis["model_probability"] = valid_probability
error_analysis["model_prediction"] = valid_prediction
error_analysis["baseline_prediction"] = baseline_probability.values

# ------------------------------------------------------------
# 3. Error categories
# ------------------------------------------------------------

error_analysis["error_type"] = "CORRECT"

error_analysis.loc[
    (error_analysis["model_prediction"] == 1) &
    (error_analysis["target"] == 0),
    "error_type"
] = "FALSE_POSITIVE"

error_analysis.loc[
    (error_analysis["model_prediction"] == 0) &
    (error_analysis["target"] == 1),
    "error_type"
] = "FALSE_NEGATIVE"

print("\nERROR COUNTS")
print("-" * 70)

display(
    error_analysis["error_type"].value_counts()
)

# ------------------------------------------------------------
# 4. Model vs baseline disagreement
# ------------------------------------------------------------

disagreements = error_analysis[
    error_analysis["model_prediction"] !=
    error_analysis["baseline_prediction"]
].copy()

print("\nMODEL VS BASELINE DISAGREEMENTS")
print("-" * 70)

print("Number of disagreements:", len(disagreements))

# ------------------------------------------------------------
# 5. Largest model false positives
# ------------------------------------------------------------

false_positives = error_analysis[
    error_analysis["error_type"] == "FALSE_POSITIVE"
].sort_values(
    "model_probability",
    ascending=False
)

print("\nTOP FALSE POSITIVES")
print("-" * 70)

display(
    false_positives[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_pageviews",
            "ga4_engaged_sessions",
            "future_gsc_clicks",
            "model_probability"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# 6. Largest model false negatives
# ------------------------------------------------------------

false_negatives = error_analysis[
    error_analysis["error_type"] == "FALSE_NEGATIVE"
].sort_values(
    "future_gsc_clicks",
    ascending=False
)

print("\nTOP FALSE NEGATIVES")
print("-" * 70)

display(
    false_negatives[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_pageviews",
            "ga4_engaged_sessions",
            "future_gsc_clicks",
            "model_probability"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# 7. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_valid,
    valid_prediction
)

print("\nCONFUSION MATRIX")
print("-" * 70)

print("Rows = actual")
print("Columns = predicted")
print(cm)

SECTION 4 — ERROR ANALYSIS

FEATURE IMPORTANCE
----------------------------------------------------------------------


,feature,importance
0,gsc_impressions,0.4875
1,gsc_clicks,0.3472
2,ga4_pageviews,0.0867
3,gsc_avg_position,0.0776
4,ga4_engaged_sessions,0.0010



ERROR COUNTS
----------------------------------------------------------------------


,count
error_type,
CORRECT,58079
FALSE_POSITIVE,6291
FALSE_NEGATIVE,1918



MODEL VS BASELINE DISAGREEMENTS
----------------------------------------------------------------------
Number of disagreements: 4881

TOP FALSE POSITIVES
----------------------------------------------------------------------


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,future_gsc_clicks,model_probability
140044,12210.0,38.0,2.794211,10.0,1.0,0.0,0.999282
31050,8569.0,18.0,6.094977,11.0,0.0,0.0,0.998669
140485,4185.0,36.0,5.361059,12.0,0.0,0.0,0.998446
44344,38111.0,56.0,29.032989,112.0,7.0,0.0,0.997979
210391,13916.0,72.0,19.054649,102.0,15.0,0.0,0.997727
46161,7489.0,85.0,12.989605,151.0,10.0,0.0,0.997695
43398,7902.0,36.0,10.822327,64.0,2.0,0.0,0.997636
44409,12620.0,13.0,23.738760,102.0,0.0,0.0,0.996591
42594,15743.0,58.0,24.025285,155.0,24.0,0.0,0.996569
45955,2492.0,15.0,13.114961,86.0,3.0,0.0,0.996418



TOP FALSE NEGATIVES
----------------------------------------------------------------------


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,future_gsc_clicks,model_probability
149125,0.0,0.0,NaN,10.0,0.0,112.0,0.303837
330399,0.0,0.0,NaN,0.0,0.0,105.0,0.073697
157504,22.0,0.0,2.545455,1.0,0.0,69.0,0.388188
314558,1.0,0.0,NaN,1.0,0.0,66.0,0.056040
329032,0.0,0.0,NaN,0.0,0.0,65.0,0.073697
32567,0.0,0.0,NaN,28.0,0.0,61.0,0.167145
120743,39.0,0.0,6.282051,0.0,0.0,60.0,0.296668
120698,46.0,0.0,5.459707,1.0,0.0,55.0,0.361414
327392,0.0,0.0,NaN,5.0,0.0,47.0,0.410785
164588,0.0,0.0,NaN,0.0,0.0,46.0,0.073697



CONFUSION MATRIX
----------------------------------------------------------------------
Rows = actual
Columns = predicted
[[47577  6291]
 [ 1918 10502]]


In [16]:
# ============================================================
# SECTION 4 — FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 70)

display(
    feature_importance.style.format({
        "importance": "{:.4f}"
    })
)

RANDOM FOREST FEATURE IMPORTANCE


,feature,importance
0,gsc_impressions,0.4875
1,gsc_clicks,0.3472
2,ga4_pageviews,0.0867
3,gsc_avg_position,0.0776
4,ga4_engaged_sessions,0.0010


### Section 4 conclusion

The Random Forest relies most strongly on Search Console signals: `gsc_impressions` accounts for 0.4875 of the feature importance and `gsc_clicks` accounts for 0.3472. Analytics pageviews and average search position contribute less, while engaged sessions contribute very little (0.0010) in this model.

The error analysis shows that the model is not always reliable when future demand changes. There were 6,291 false positives and 1,918 false negatives on the validation set. Some false negatives had very weak March search signals but generated substantial April clicks, including cases with more than 100 future clicks. Conversely, some false positives had strong March performance but received zero April clicks.

These errors suggest that historical performance signals are useful but cannot fully capture changes in search demand, seasonality, search intent, or content quality. The model should therefore be treated as decision-support rather than as proof that a page should be refreshed.

## Self-check

- [x] Every section above is filled with the required markdown explanations and supporting code/results.
- [x] The notebook runs from top to bottom without errors using Runtime → Run all.
- [x] No client names, URLs, or private queries are included.
- [x] Claims are stated carefully using words such as observed, measured, directional, and decision-support.
- [x] The Random Forest is compared with the Week-4 baseline using the same validation data and the same evaluation metrics.
- [x] The model's errors and important signals are interpreted rather than relying only on the headline metric.
- [x] The notebook has been committed and pushed to my repository under `work/notebooks/w05_model.ipynb`, and the repository URL has been submitted on the assignment card.